# Parte 5 — Persistência com SQLite

**Objetivo desta parte:** parar de depender de arquivos CSV soltos e persistir os
dados em um banco relacional de verdade. Vamos criar duas tabelas — `clima_raw`
(horária, tratada) e `clima_diario` (agregada) —, praticar `to_sql()` do pandas e
depois **`INSERT` manual com `sqlite3` puro**, para enxergar o que o pandas
abstrai por baixo dos panos. Fechamos repetindo tudo com **SQLAlchemy**.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
DB_PATH = DATA_DIR / "clima.db"

horario = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv", parse_dates=["datetime"])
diario = pd.read_csv(PROCESSED_DIR / "clima_diario.csv", parse_dates=["data"])

diario_para_db = diario[[
    "cidade", "data", "temp_media", "temp_min", "temp_max", "umidade_media",
    "precipitacao_total", "vento_medio", "categoria_temp", "categoria_chuva",
    "media_movel_3d", "media_movel_7d", "indice_conforto_c",
]].copy()


## 1. Conexão e criação das tabelas (SQL puro)

Definimos **chave primária composta** em ambas as tabelas: `(cidade, datetime)`
para o dado horário e `(cidade, data)` para o diário. Uma leitura é identificada
unicamente pela combinação de cidade + momento no tempo — nem `cidade` nem
`datetime` sozinhos bastariam.

Isso é o que vai permitir fazer **UPSERT** depois: se o pipeline rodar de novo
sobre o mesmo período, a chave primária impede duplicatas.


In [2]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS clima_raw (
    cidade TEXT NOT NULL,
    datetime TEXT NOT NULL,
    temp_c REAL,
    umidade_pct REAL,
    precipitacao_mm REAL,
    vento_kmh REAL,
    PRIMARY KEY (cidade, datetime)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS clima_diario (
    cidade TEXT NOT NULL,
    data TEXT NOT NULL,
    temp_media REAL,
    temp_min REAL,
    temp_max REAL,
    umidade_media REAL,
    precipitacao_total REAL,
    vento_medio REAL,
    categoria_temp TEXT,
    categoria_chuva TEXT,
    media_movel_3d REAL,
    media_movel_7d REAL,
    indice_conforto_c REAL,
    PRIMARY KEY (cidade, data)
)
''')

conn.commit()
print("Tabelas criadas (ou já existentes).")


Tabelas criadas (ou já existentes).


## 2. `to_sql()` do pandas

A forma mais rápida de jogar um DataFrame inteiro para o banco. Repare que
usamos uma tabela **separada** (`clima_raw_pandas`) e `if_exists="replace"`: o
`to_sql` não sabe nada sobre a nossa chave primária composta, então rodá-lo duas
vezes com `if_exists="append"` sobre a tabela `clima_raw` (que tem PK) quebraria
por violação de chave, e com `"replace"` ele recriaria a tabela do zero, perdendo
o `PRIMARY KEY` que definimos manualmente.

Isso é exatamente o problema que o UPSERT manual (próxima seção) resolve.


In [3]:
horario.to_sql("clima_raw_pandas", conn, if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS total_linhas FROM clima_raw_pandas", conn)


,total_linhas
0,3720


## 3. `INSERT` manual com `sqlite3` puro + UPSERT

Aqui vemos o que o `to_sql` esconde: a conversão linha a linha de um DataFrame em
tuplas, o `executemany`, e a cláusula `ON CONFLICT ... DO UPDATE` que faz o
upsert — se a chave `(cidade, datetime)` já existe, atualiza os valores; senão,
insere uma linha nova.


In [4]:
def upsert_clima_raw(df: pd.DataFrame, conn: sqlite3.Connection) -> None:
    linhas = list(
        df[["cidade", "datetime", "temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]]
        .assign(datetime=lambda d: d["datetime"].astype(str))
        .itertuples(index=False, name=None)
    )

    conn.executemany(
        '''
        INSERT INTO clima_raw (cidade, datetime, temp_c, umidade_pct, precipitacao_mm, vento_kmh)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT (cidade, datetime) DO UPDATE SET
            temp_c = excluded.temp_c,
            umidade_pct = excluded.umidade_pct,
            precipitacao_mm = excluded.precipitacao_mm,
            vento_kmh = excluded.vento_kmh
        ''',
        linhas,
    )
    conn.commit()


upsert_clima_raw(horario, conn)
pd.read_sql("SELECT COUNT(*) AS total_linhas FROM clima_raw", conn)


,total_linhas
0,3720


In [5]:
# Rodando de novo o mesmo upsert sobre o mesmo dado: a contagem de linhas
# não deve mudar (nem duplicar, nem dar erro de chave primária).
upsert_clima_raw(horario, conn)
pd.read_sql("SELECT COUNT(*) AS total_linhas FROM clima_raw", conn)


,total_linhas
0,3720


## 4. O mesmo padrão para `clima_diario`


In [6]:
def upsert_clima_diario(df: pd.DataFrame, conn: sqlite3.Connection) -> None:
    colunas = [
        "cidade", "data", "temp_media", "temp_min", "temp_max", "umidade_media",
        "precipitacao_total", "vento_medio", "categoria_temp", "categoria_chuva",
        "media_movel_3d", "media_movel_7d", "indice_conforto_c",
    ]
    linhas = list(
        df[colunas].assign(data=lambda d: d["data"].astype(str)).itertuples(index=False, name=None)
    )
    placeholders = ", ".join(["?"] * len(colunas))
    colunas_sql = ", ".join(colunas)
    atualizacoes = ", ".join(f"{c} = excluded.{c}" for c in colunas if c not in ("cidade", "data"))

    conn.executemany(
        f'''
        INSERT INTO clima_diario ({colunas_sql})
        VALUES ({placeholders})
        ON CONFLICT (cidade, data) DO UPDATE SET
            {atualizacoes}
        ''',
        linhas,
    )
    conn.commit()


upsert_clima_diario(diario_para_db, conn)
upsert_clima_diario(diario_para_db, conn)  # rodar 2x para provar a idempotência

pd.read_sql("SELECT COUNT(*) AS total_linhas FROM clima_diario", conn)


,total_linhas
0,155


## 5. Consultas (`SELECT`) e recuperação dos dados

Algumas formas de ler de volta: com `pandas.read_sql` (retorna DataFrame direto,
ótimo para análise) e com o `cursor` puro (retorna tuplas, mais controle sobre o
que fazer com cada linha).


In [7]:
# Temperatura média por cidade, direto via SQL (em vez de pandas)
pd.read_sql(
    '''
    SELECT cidade, ROUND(AVG(temp_media), 1) AS temp_media_mes
    FROM clima_diario
    GROUP BY cidade
    ORDER BY temp_media_mes DESC
    ''',
    conn,
)


,cidade,temp_media_mes
0,manaus,27.1
1,rio_de_janeiro,27.0
2,recife,26.9
3,porto_alegre,25.2
4,sao_paulo,22.7


In [8]:
# Filtro por data e por cidade
pd.read_sql(
    "SELECT * FROM clima_diario WHERE cidade = ? AND data >= ? ORDER BY data",
    conn,
    params=("recife", "2025-01-25"),
)


,cidade,data,temp_media,temp_min,temp_max,umidade_media,precipitacao_total,vento_medio,categoria_temp,categoria_chuva,media_movel_3d,media_movel_7d,indice_conforto_c
0,recife,2025-01-25,26.608333,24.2,29.3,78.187500,3.4,8.845833,quente,chuvoso,27.180556,27.079762,26.6
1,recife,2025-01-26,25.737500,23.5,28.4,83.083333,10.6,8.008333,ameno,chuvoso,26.691667,27.017262,25.7
2,recife,2025-01-27,25.766667,23.6,29.0,86.791667,12.3,7.191667,ameno,chuvoso,26.037500,26.850595,25.8
3,recife,2025-01-28,25.491667,22.9,27.5,86.541667,17.3,6.475000,ameno,chuvoso,25.665278,26.562500,25.5
4,recife,2025-01-29,25.641667,23.9,27.5,85.750000,2.5,8.137500,ameno,chuvoso,25.633333,26.311310,25.6
5,recife,2025-01-30,26.800000,24.0,29.6,79.500000,3.7,9.341667,quente,chuvoso,25.977778,26.253571,26.8
6,recife,2025-01-31,27.420833,25.0,30.1,71.541667,0.2,10.029167,quente,seco,26.620833,26.209524,30.0


In [9]:
# JOIN entre a tabela horária e a diária: horas de um dia específico junto com
# o resumo daquele dia
cursor.execute(
    '''
    SELECT r.cidade, r.datetime, r.temp_c, d.temp_media, d.categoria_temp
    FROM clima_raw AS r
    JOIN clima_diario AS d
        ON r.cidade = d.cidade AND DATE(r.datetime) = d.data
    WHERE r.cidade = 'manaus' AND DATE(r.datetime) = '2025-01-15'
    ORDER BY r.datetime
    LIMIT 5
    '''
)
cursor.fetchall()


[('manaus', '2025-01-15 00:00:00', 26.7, 27.75208333333333, 'quente'),
 ('manaus', '2025-01-15 01:00:00', 26.5, 27.75208333333333, 'quente'),
 ('manaus', '2025-01-15 02:00:00', 26.2, 27.75208333333333, 'quente'),
 ('manaus', '2025-01-15 03:00:00', 26.0, 27.75208333333333, 'quente'),
 ('manaus', '2025-01-15 04:00:00', 25.8, 27.75208333333333, 'quente')]

In [10]:
conn.close()


## 6. Repetindo com SQLAlchemy

SQLAlchemy adiciona uma camada de abstração sobre o `sqlite3`: em vez de escrever
SQL cru para criar tabelas, descrevemos a estrutura em Python (`Table`,
`Column`...) através do **Core** (não o ORM completo, que seria um conteúdo à
parte). O `engine` gerencia a conexão com o banco.


In [11]:
from sqlalchemy import (
    Column,
    Float,
    MetaData,
    String,
    Table,
    create_engine,
    func,
    select,
)
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert

engine = create_engine(f"sqlite:///{DB_PATH}")
metadata = MetaData()

clima_raw_sa = Table(
    "clima_raw",
    metadata,
    Column("cidade", String, primary_key=True),
    Column("datetime", String, primary_key=True),
    Column("temp_c", Float),
    Column("umidade_pct", Float),
    Column("precipitacao_mm", Float),
    Column("vento_kmh", Float),
)

# As tabelas já existem (criadas na Seção 1) — create_all com checkfirst=True
# não recria nada, só confirma que a estrutura Python bate com o banco.
metadata.create_all(engine, checkfirst=True)


### `to_sql` via engine do SQLAlchemy

O mesmo `to_sql` do pandas, mas passando um `engine` do SQLAlchemy em vez de uma
conexão `sqlite3` crua — é a forma recomendada pelo próprio pandas hoje em dia,
e funciona com qualquer banco suportado pelo SQLAlchemy (Postgres, MySQL...) sem
mudar uma linha de código.


In [12]:
horario.to_sql("clima_raw_pandas", engine, if_exists="replace", index=False)

with engine.connect() as conn_sa:
    total = conn_sa.execute(select(func.count()).select_from(clima_raw_sa)).scalar()
    print("Linhas em clima_raw (tabela com PK, via Core):", total)


Linhas em clima_raw (tabela com PK, via Core): 3720


### UPSERT com SQLAlchemy Core

O dialeto SQLite do SQLAlchemy expõe `on_conflict_do_update`, o equivalente
tipado do `ON CONFLICT ... DO UPDATE` que escrevemos à mão na Seção 3.


In [13]:
def upsert_clima_raw_sqlalchemy(df: pd.DataFrame, engine) -> None:
    registros = (
        df[["cidade", "datetime", "temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]]
        .assign(datetime=lambda d: d["datetime"].astype(str))
        .to_dict(orient="records")
    )

    stmt = sqlite_upsert(clima_raw_sa).values(registros)
    stmt = stmt.on_conflict_do_update(
        index_elements=["cidade", "datetime"],
        set_={
            "temp_c": stmt.excluded.temp_c,
            "umidade_pct": stmt.excluded.umidade_pct,
            "precipitacao_mm": stmt.excluded.precipitacao_mm,
            "vento_kmh": stmt.excluded.vento_kmh,
        },
    )

    with engine.begin() as conn_sa:
        conn_sa.execute(stmt)


upsert_clima_raw_sqlalchemy(horario, engine)
upsert_clima_raw_sqlalchemy(horario, engine)  # idempotente, igual à Seção 3

with engine.connect() as conn_sa:
    print(conn_sa.execute(select(func.count()).select_from(clima_raw_sa)).scalar())


3720


### `SELECT` via SQLAlchemy Core e via pandas

`select(...)` monta a query de forma programática (útil quando os filtros vêm
dinamicamente, ex.: de uma API); `pd.read_sql` com o `engine` continua sendo o
caminho mais direto quando o destino é um DataFrame.


In [14]:
query = (
    select(
        clima_raw_sa.c.cidade,
        func.round(func.avg(clima_raw_sa.c.temp_c), 1).label("temp_media"),
    )
    .group_by(clima_raw_sa.c.cidade)
    .order_by(func.avg(clima_raw_sa.c.temp_c).desc())
)

pd.read_sql(query, engine)


,cidade,temp_media
0,manaus,27.1
1,rio_de_janeiro,27.0
2,recife,26.9
3,porto_alegre,25.2
4,sao_paulo,22.7


In [15]:
pd.read_sql("SELECT * FROM clima_diario WHERE categoria_temp = 'quente' ORDER BY indice_conforto_c DESC LIMIT 10", engine)


,cidade,data,temp_media,temp_min,temp_max,umidade_media,precipitacao_total,vento_medio,categoria_temp,categoria_chuva,media_movel_3d,media_movel_7d,indice_conforto_c
0,rio_de_janeiro,2025-01-21,31.691667,27.9,37.0,56.708333,0.3,6.904167,quente,seco,30.400000,28.843452,35.7
1,rio_de_janeiro,2025-01-20,30.129167,25.7,36.4,69.375000,1.6,5.645833,quente,chuvoso,30.015278,27.813690,35.5
2,rio_de_janeiro,2025-01-18,30.537500,24.1,38.0,65.083333,0.1,7.220833,quente,seco,28.637500,26.191071,35.4
3,rio_de_janeiro,2025-01-22,30.212500,26.9,34.3,66.041667,0.0,5.820833,quente,seco,30.677778,29.617857,34.9
4,rio_de_janeiro,2025-01-19,29.379167,25.2,36.2,71.625000,0.2,6.045833,quente,seco,29.555556,26.913690,34.2
5,rio_de_janeiro,2025-01-25,30.056250,25.5,34.6,63.791667,0.0,6.000000,quente,seco,28.765972,29.672917,34.0
6,rio_de_janeiro,2025-01-26,29.060417,26.5,33.4,70.666667,5.3,6.162500,quente,chuvoso,29.000000,29.627381,33.3
7,rio_de_janeiro,2025-01-17,28.750000,24.4,33.9,72.145833,0.0,8.508333,quente,seco,26.722222,25.304167,32.9
8,rio_de_janeiro,2025-01-23,28.358333,25.7,32.3,76.000000,0.0,6.433333,quente,seco,30.087500,29.865476,32.7
9,manaus,2025-01-16,28.066667,26.0,31.4,79.291667,0.0,4.404167,quente,seco,27.893750,27.787500,32.5


In [16]:
engine.dispose()


## Resumo: quando usar cada abordagem

| Abordagem | Quando usar |
|---|---|
| `df.to_sql(..., if_exists="replace")` | Prototipagem rápida, recarga completa da tabela, sem preocupação com chave primária |
| `sqlite3` puro + `ON CONFLICT` | Entender exatamente o que acontece linha a linha; controle total sobre o SQL |
| SQLAlchemy Core + `on_conflict_do_update` | Mesma lógica de upsert, mas portável entre bancos (trocar SQLite por Postgres exige só trocar a connection string) |
